# Μάθημα 07 - Πρότυπο Σχεδιασμού Προγραμματισμού

Αυτό το σημειωματάριο παρουσιάζει το **Πρότυπο Σχεδιασμού Προγραμματισμού** για πράκτορες AI χρησιμοποιώντας το Microsoft Agent Framework.
Θα μάθετε πώς να σπάτε πολύπλοκα αιτήματα ταξιδιού σε δομημένα υπο-καθήκοντα, να τα αναθέτετε σε εξειδικευμένους πράκτορες,
και να εκτελείτε το προκύπτον σχέδιο — όλα χρησιμοποιώντας δομημένη έξοδο που τροφοδοτείται από μοντέλα Pydantic.


## Ρύθμιση


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os, asyncio
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Ανάλυση Εργασιών

Η ανάλυση εργασιών είναι ο πυρήνας του σχεδιαστικού προτύπου του προγραμματισμού. Αντί να ζητούμε από έναν μόνο πράκτορα να χειριστεί ένα πολύπλοκο αίτημα από την αρχή μέχρι το τέλος, διασπούμε το πρόβλημα σε μικρότερες, καλά ορισμένες **υποεργασίες**. Κάθε υποεργασία ανατίθεται σε έναν ειδικό πράκτορα (π.χ., πτήσεις, ξενοδοχεία, δραστηριότητες) με σαφείς προτεραιότητες και σειρά εξαρτήσεων.

Αυτή η προσέγγιση παρέχει αρκετά οφέλη:
- **Σαφήνεια**: κάθε υποεργασία έχει μία μόνο ευθύνη.
- **Παράλληλη εκτέλεση**: ανεξάρτητες υποεργασίες μπορούν να εκτελεστούν ταυτόχρονα.
- **Αξιοπιστία**: οι αποτυχίες περιορίζονται σε μεμονωμένες υποεργασίες.
- **Παρακολούθηση προϋπολογισμού**: τα κόστη εκτιμώνται ανά υποεργασία και συνοψίζονται.


In [ ]:
class TravelSubTask(BaseModel):
    task_id: int
    description: str
    assigned_agent: str  # "flight_agent", "hotel_agent", "activity_agent"
    priority: str  # "high", "medium", "low"
    dependencies: list[int] = []


class TravelPlan(BaseModel):
    destination: str
    trip_duration_days: int
    subtasks: list[TravelSubTask]
    total_estimated_budget_usd: int
    notes: str

## Δημιουργία Πράκτορα Σχεδιασμού με Δομημένη Έξοδο

Ο πράκτορας σχεδιασμού λειτουργεί ως **συντονιστής υποδοχής**. Δεδομένου ενός αιτήματος ταξιδιού υψηλού επιπέδου, παράγει ένα δομημένο `TravelPlan` — διασπά το αίτημα σε υποεργασίες, ορίζει προτεραιότητες και εντοπίζει εξαρτήσεις ώστε ένας κονσιέρζ ή επίπεδο εκτέλεσης να μπορεί να αναλάβει την εργασία.


In [ ]:
planning_agent = await provider.create_agent(
    name="TravelPlanner",
    instructions="""You are a travel planning agent. When given a travel request:
1. Break it into specific subtasks (flights, hotels, activities, logistics)
2. Assign each subtask to the appropriate specialist agent
3. Set priorities and identify dependencies between tasks
4. Estimate the total budget""",
)

result = await planning_agent.run(
    "Plan a 7-day trip to Paris for a couple interested in art, cuisine, and history. Budget around $5000.",
    )
if result:
    plan = result
    print(f"Destination: {plan.destination}")
    print(f"Duration: {plan.trip_duration_days} days")
    print(f"Budget: ${plan.total_estimated_budget_usd}")
    print(f"\nSubtasks:")
    for task in plan.subtasks:
        print(f"  [{task.priority}] {task.task_id}. {task.description} → {task.assigned_agent}")

## Εκτέλεση ενός Σχεδίου με Εξειδικευμένα Εργαλεία

Μόλις ο υπάλληλος υποδοχής καταρτίσει ένα δομημένο σχέδιο, ο **υπάλληλος υποδοχής υπηρεσιών** το εκτελεί.
Κάθε εξειδικευμένο εργαλείο διαχειρίζεται μία κατηγορία υποεργασιών (πτήσεις, ξενοδοχεία, δραστηριότητες). Ο υπάλληλος υποδοχής διατρέχει τις υποεργασίες του σχεδίου με βάση τη σειρά εξάρτησης και αναθέτει κάθε μία στο κατάλληλο εργαλείο.


In [ ]:
@tool
def book_flight(
    destination: Annotated[str, "The destination city"],
    departure_date: Annotated[str, "Departure date (YYYY-MM-DD)"],
    return_date: Annotated[str, "Return date (YYYY-MM-DD)"],
) -> str:
    """Search and book flights for the trip."""
    return f"Flight booked to {destination}: {departure_date} → {return_date}, confirmation #FLT-{hash(destination) % 10000:04d}"


@tool
def reserve_hotel(
    city: Annotated[str, "The city for the hotel"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"],
) -> str:
    """Reserve a hotel room in the destination city."""
    return f"Hotel reserved in {city}: {check_in} to {check_out} for {guests} guests, confirmation #HTL-{hash(city) % 10000:04d}"


@tool
def book_activity(
    activity_name: Annotated[str, "Name of the activity or tour"],
    date: Annotated[str, "Date of the activity (YYYY-MM-DD)"],
    participants: Annotated[int, "Number of participants"],
) -> str:
    """Book a tour, museum visit, or other activity."""
    return f"Activity booked: {activity_name} on {date} for {participants} people, confirmation #ACT-{hash(activity_name) % 10000:04d}"


# Concierge agent that executes the plan using specialist tools
concierge_agent = await provider.create_agent(
    name="Concierge",
    instructions="""You are a travel concierge executing a structured travel plan.
Use the available tools to fulfil each subtask. Work through the subtasks in order,
respecting dependencies. Summarise the results when finished.""",
    tools=[book_flight, reserve_hotel, book_activity],
)

# Build a prompt from the plan produced above
if result.value:
    subtask_lines = "\n".join(
        f"- [{t.priority}] {t.task_id}. {t.description} (agent: {t.assigned_agent}, deps: {t.dependencies})"
        for t in plan.subtasks
    )
    execution_prompt = (
        f"Execute the following travel plan for {plan.destination} "
        f"({plan.trip_duration_days} days, ${plan.total_estimated_budget_usd} budget):\n"
        f"{subtask_lines}"
    )

    exec_response = await concierge_agent.run(execution_prompt)
    print(exec_response)

## Περίληψη

Σε αυτό το μάθημα μάθατε το **Σχεδιαστικό Πρότυπο Προγραμματισμού** για πράκτορες AI:

1. **Αποσύνθεση Εργασιών** — Ένας πράκτορας προγραμματισμού υποδοχής διασπά ένα σύνθετο αίτημα ταξιδιού σε
   δομημένες υποεργασίες χρησιμοποιώντας μοντέλα Pydantic, αναθέτοντας κάθε μία σε ειδικό πράκτορα με προτεραιότητες
   και εξαρτήσεις.
2. **Δομημένη Έξοδος** — Με τη χρήση `response_format`, ο πράκτορας επιστρέφει ένα επαληθευμένο
   αντικείμενο `TravelPlan` αντί για ελεύθερο κείμενο, καθιστώντας την επεξεργασία στη συνέχεια πιο αξιόπιστη.
3. **Εκτέλεση Σχεδίου** — Ένας πράκτορας υποδοχής διατρέχει τις υποεργασίες χρησιμοποιώντας ειδικά εργαλεία
   (`book_flight`, `reserve_hotel`, `book_activity`) για να εκτελέσει το σχέδιο και να αναφέρει τα αποτελέσματα.

Αυτό το πρότυπο διαχωρίζει το *τι να κάνει* (προγραμματισμός) από το *πώς να το κάνει* (εκτέλεση), καθιστώντας τους πράκτορες
πιο αρθρωτούς, δοκιμαστικούς και ευκολότερους στην επέκταση.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Αποποίηση Ευθυνών**:
Αυτό το έγγραφο έχει μεταφραστεί χρησιμοποιώντας την υπηρεσία μετάφρασης με τεχνητή νοημοσύνη [Co-op Translator](https://github.com/Azure/co-op-translator). Παρόλο που προσπαθούμε για ακρίβεια, παρακαλούμε να γνωρίζετε ότι οι αυτοματοποιημένες μεταφράσεις μπορεί να περιέχουν σφάλματα ή ανακρίβειες. Το πρωτότυπο έγγραφο στη γλώσσα του θεωρείται η αυθεντική πηγή. Για κρίσιμες πληροφορίες, συνιστάται επαγγελματική μετάφραση από άνθρωπο. Δεν φέρουμε ευθύνη για τυχόν παρεξηγήσεις ή λανθασμένες ερμηνείες που προκύπτουν από τη χρήση αυτής της μετάφρασης.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
